# Ecosistema de procesamiento paralelo para datos masivos

## Módulo 3. Lección 1: El paradigma de programación MapReduce: hadoop-streaming

### Ejercicios resueltos 03

**IMPORTANTE**

Ejecuta las celdas en orden para evitar errores de ejecución. Esto garantiza que todos los archivos necesarios se creen y se ubiquen correctamente a lo largo del proceso.

#### **Obtención de los datos**

In [2]:
# creamos la carpeta para guardar los ficheros
! mkdir -p /media/notebooks/argentina

# Descargamos el fichero 
! wget https://github.com/Vega90/datasets/raw/main/convocatorias-2020.csv \
-O /media/notebooks/argentina/convocatorias-2020.csv

--2024-08-28 08:32:23--  https://github.com/Vega90/datasets/raw/main/convocatorias-2020.csv
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Vega90/datasets/main/convocatorias-2020.csv [following]
--2024-08-28 08:32:24--  https://raw.githubusercontent.com/Vega90/datasets/main/convocatorias-2020.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5675992 (5.4M) [text/plain]
Saving to: ‘/media/notebooks/argentina/convocatorias-2020.csv’

/media/notebooks/ar 100%[===================>]   5.41M  14.4MB/s    in 0.4s    

2024-08-28 08:32:24 (14.4 MB/s) - ‘/media/notebooks/argentina/convocatorias-2020.c

In [3]:
# subimos los ficheros a HDFS 
! hdfs dfs -mkdir /argentina
! hdfs dfs -put /media/notebooks/argentina/convocatorias-2020.csv /argentina
! hdfs dfs -ls /argentina

Found 1 items
-rw-r--r--   3 root supergroup    5675992 2024-08-28 08:32 /argentina/convocatorias-2020.csv


#### **Creación carpeta para los ejercicios**

In [4]:
import os
# creamos la carpeta para los ejercicios
! mkdir -p /media/notebooks/argentina/mapreduce
# nos vamos a esa carpeta y comprobamos donde estamos
os.chdir("/media/notebooks/argentina/mapreduce")
! pwd

/media/notebooks/argentina/mapreduce


#### **Dataframe de pandas para comprender los datos y comparar la salida**

Utilizaremos un dataframe de pandas para ayudarnos a entender los datos y también podemos comprobar los ejercicios comparando la salida dada y la salida esperada.

In [5]:
import pandas as pd

# evitamos la notación científica en todos los ejercicios
pd.set_option('display.float_format', '{:.2f}'.format)

# dataframe convocatorias
columnas = [
             'numero_procedimiento', 'nro_saf', 'descripcion_saf', 'nro_uoc', 
             'descripcion_uoc','tipo_de_procedimiento', 'modalidad', 'apartado_directa', 
             'ejercicio', 'fecha_de_publicacion', 'fecha_de_apertura', 'etapa', 'alcance', 
             'nombre_del_procedimiento', 'objeto_del_procedimiento', 'monto_estimado', 'tipo_de_operacion'
           ]

df_convocatorias = pd.read_csv('/media/notebooks/argentina/convocatorias-2020.csv', names=columnas) 
df_convocatorias.head(3)

,numero_procedimiento,nro_saf,descripcion_saf,nro_uoc,descripcion_uoc,tipo_de_procedimiento,modalidad,apartado_directa,ejercicio,fecha_de_publicacion,fecha_de_apertura,etapa,alcance,nombre_del_procedimiento,objeto_del_procedimiento,monto_estimado,tipo_de_operacion
0,40/51-0159-LPR19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Licitacion Privada,Sin Modalidad,NaN,2019,02/01/2020 08:00:00 a.m.,10/01/2020 10:00:00 a.m.,Única,Nacional,“Adquisición de Elementos de Librería para el ...,“Adquisición de Elementos de Librería para el ...,1957142.84,Proceso de Compra
1,22-0041-CDI19,250,250 - Caja de Retiros Jubilaciones y Pensiones...,22,22 - Dpto de Compras y Suministros - Caja de R...,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 12:00:00 p.m.,13/01/2020 09:00:00 a.m.,Única,Nacional,Servicio de seguridad de información sensible ...,Servicio de seguridad de información sensible ...,119915.25,Proceso de Compra
2,40/51-0918-CDI19,381,381 - Estado Mayor General de La Fuerza Aérea,40,Departamento Contrataciones Córdoba - UOC 40/51,Contratación Directa,Sin Modalidad,Apartado 1: Compulsa Abreviada Por Monto,2019,02/01/2020 12:00:00 p.m.,13/01/2020 10:00:00 a.m.,Única,Nacional,“Servicio Médico Asistencial para Organismos V...,“Servicio Médico Asistencial para Organismos V...,99000.00,Proceso de Compra


#### **Ejercicio 08.** Tipos de procedimientos. Transparencia de compras.

Este ejercicio va a evaluar la transparencia en el sistema de compras del Estado procesados en el sistema COMPRAR. Para ello, debemos tener claro el criterio que seguiremos: El sistema es más transparente si se utiliza el procedimiento de licitación pública. Para contestar a ésta pregunta calcularemos el porcentaje de cada tipo de procedimiento.

En la fase Map tendremos que obtener los campos que nos interesan. En este caso el tipo de contratación. Escribiremos este campo como clave seguido de un '1' que nos servirá para contar el total de cada tipo de procedimiento y agrupar resultados.

**Proceso MAP 08**

In [6]:
%%writefile 08_mapper.py
#!/usr/bin/env python3

import sys

# entrada estandar STDIN
for linea in sys.stdin:
  # eliminamos espacios y dividimos
  campos = linea.strip().split(',')

  # nos quedamos con los campos que queremos
  tipo_procedimiento  = campos[5]
  print(f"{tipo_procedimiento}\t1")

Writing 08_mapper.py


**Proceso REDUCE 08**

In [24]:
%%writefile 08_reducer.py
#!/usr/bin/env python3
import sys
from collections import defaultdict

# diccionario para los tipos 
dict_totales = defaultdict(int)
procedimientos_totales = 0

# procesamos cada linea de entrada
for linea in sys.stdin:
    contenido = linea.strip().split('\t')
    
    if len(contenido) != 2:
        continue

    procedimiento, numero = contenido
    numero = int(numero)

    # guardamos los procedimientos y sumamos totales
    dict_totales[procedimiento] += numero
    procedimientos_totales += numero

#ordenamos por los valores de forma descendente (de mayor a menor)
dict_ordenado = dict(sorted(dict_totales.items(), key=lambda item: item[1], reverse=True))

# calculamos porcentajes e imprimimos los resultados
for procedimiento, numero in dict_ordenado.items():
    porcentaje = round((numero / procedimientos_totales) * 100, 2)
    print(f"{procedimiento}:\t{porcentaje}")

Overwriting 08_reducer.py


In [25]:
# ejecutamos nuestro map-reduce y vemos el resultado desde la consola 
! cat /media/notebooks/argentina/convocatorias-2020.csv \
| python3 08_mapper.py | sort | python3 08_reducer.py > salida-08

! cat salida-08

Contratación Directa:	49.94
Licitacion Privada:	38.2
Licitacion Pública:	10.94
Subasta:	0.47
Concurso Privado:	0.27
Concurso Público:	0.17


**Ejecución Hadoop Streaming 08**

In [30]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 08_mapper.py,08_reducer.py \
-mapper 08_mapper.py \
-reducer 08_reducer.py \
-input /argentina/convocatorias-2020.csv \
-output /argentina/mapreduce/08_salida

packageJobJar: [/tmp/hadoop-unjar3347147290728673892/] [] /tmp/streamjob2069858118737928105.jar tmpDir=null
2024-08-28 08:41:33,632 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-28 08:41:33,892 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-28 08:41:34,545 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1724833657149_0008
2024-08-28 08:41:35,752 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-28 08:41:36,540 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-28 08:41:37,257 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1724833657149_0008
2024-08-28 08:41:37,257 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-28 08:41:37,788 INFO conf.Configuration: resource-types.xml not found
2024-08-28 08:41:37,789 INFO resource.ResourceUtils

In [31]:
# Visualizamos la salida del ejercicio 8
! hdfs dfs -cat /argentina/mapreduce/08_salida/part-00000 

# comprobamos en el dataframe los porcentajes
totales = df_convocatorias['tipo_de_procedimiento'].value_counts()

# calculamos los porcentajes
porcentaje_tipos = round((totales / len(df_convocatorias)) * 100, 2)

print(f"\nValores en el dataframe (en porcentaje):\n\n{porcentaje_tipos}")

Contratación Directa:	49.94
Licitacion Privada:	38.2
Licitacion Pública:	10.94
Subasta:	0.47
Concurso Privado:	0.27
Concurso Público:	0.17

Valores en el dataframe (en porcentaje):

tipo_de_procedimiento
Contratación Directa   49.89
Licitacion Privada     38.17
Licitacion Pública     10.93
Subasta                 0.47
Concurso Privado        0.27
Concurso Público        0.17
Name: count, dtype: float64


#### **Ejercicio 09.** Cálculo del valor Medio. Presupuesto oficial estimado para obras nacionales e internacionales.

En el ejercicio número 9 vamos a calcular cuál es el presupuesto medio que se emplea en las obras públicas para aquellas que son nacionales y las que son internacionales. Por lo tanto, debemos separar ambos valores.

**Proceso MAP 09**

In [11]:
%%writefile 09_mapper.py
#!/usr/bin/env python3

import sys

# entrada estandar STDIN
for linea in sys.stdin:
  # eliminamos espacios y dividimos
  campos = linea.strip().split(',')
  # nos quedamos con los campos que queremos
  alcance, monto_estimado = campos[12], campos[15]

  # ignoramos valores de monto 0
  if monto_estimado != '0':         
      print(f"{alcance}\t{monto_estimado}")

Writing 09_mapper.py


**Proceso Reduce 09**

In [12]:
%%writefile 09_reducer.py
#!/usr/bin/env python3
import sys
from collections import defaultdict

# diccionarios para el total de montos y el conteo de alcances
total_monto = defaultdict(float)
totales = defaultdict(int)

# por cada linea leemos la informacion del mapper
for linea in sys.stdin:
    alcance, monto_estimado = linea.strip().split('\t')
    
    if not monto_estimado.replace('.', '', 1).isdigit():
        continue

    monto_estimado = float(monto_estimado)
    
    # el monto estimado y vamos contando el total
    total_monto[alcance] += monto_estimado
    totales[alcance] += 1

# calculamos e imprimimos el presupuesto medio
for alcance in total_monto:
    promedio = round(total_monto[alcance] / totales[alcance], 2)
    print(f"Presupuesto medio para {alcance}:\t{promedio}")

Writing 09_reducer.py


In [13]:
# ejecutamos nuestro map-reduce desde la consola
! cat /media/notebooks/argentina/convocatorias-2020.csv \
| python3 09_mapper.py | sort | python3 09_reducer.py > salida-09

! cat salida-09

Presupuesto medio para Internacional:	12112060.49
Presupuesto medio para Nacional:	8868861.54


**Ejecución Hadoop Streaming 09**

In [14]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 09_mapper.py,09_reducer.py \
-mapper 09_mapper.py \
-reducer 09_reducer.py \
-input /argentina/convocatorias-2020.csv \
-output /argentina/mapreduce/09_salida

packageJobJar: [/tmp/hadoop-unjar2311105442875292328/] [] /tmp/streamjob5435059549137335215.jar tmpDir=null
2024-08-28 08:33:52,382 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-28 08:33:52,934 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-28 08:33:53,582 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1724833657149_0002
2024-08-28 08:33:54,565 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-28 08:33:55,183 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-28 08:33:55,975 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1724833657149_0002
2024-08-28 08:33:55,975 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-28 08:33:56,714 INFO conf.Configuration: resource-types.xml not found
2024-08-28 08:33:56,716 INFO resource.ResourceUtils

In [15]:
# Visualizamos la salida del ejercicio 9
! hdfs dfs -cat /argentina/mapreduce/09_salida/part-00000 

# eliminamos los que tienen valor 0
df_filtrado = df_convocatorias[df_convocatorias['monto_estimado'] != 0]
# agrupamos y calculamos la media
promedio_monto_por_alcance = df_filtrado.groupby('alcance')['monto_estimado'].mean()
print(f"\nValores en el dataframe:\n\n{promedio_monto_por_alcance}")

Presupuesto medio para Internacional:	12112060.49
Presupuesto medio para Nacional:	8868861.54

Valores en el dataframe:

alcance
Internacional   12112060.49
Nacional         8868861.54
Name: monto_estimado, dtype: float64


#### **Ejercicio 10.** Longitud de cadenas. Clasificación de nombres por su longitud

Este ejercicio nos servirá para practicar algo que todavía no hemos abordado: contar cadenas de texto. El objetivo es procesar una cadena de texto, en este caso, el nombre de procedimiento y clasificarla según su longitud. Además imprimiremos los nombres que entren dentro de la categoría 'largo'.

**Proceso MAP 10**

In [16]:
%%writefile 10_mapper.py
#!/usr/bin/env python3
import sys

# entrada estandar STDIN
for linea in sys.stdin:
    # eliminamos espacios y dividimos
    campos = linea.strip().split(',')
    
    # nos quedamos con los campos que queremos
    nombre_procedimiento = campos[13]
    longitud = len(nombre_procedimiento)
    
    # clasificamos por longitud
    if longitud <= 50:
        categoria = 'corto'
    elif longitud <= 100:
        categoria = 'medio'
    else:
        categoria = 'largo'
    
    # imprimimos la categoria con un conteo de 1 y el nombre del procedimiento
    print(f"{categoria}\t1\t{nombre_procedimiento}")

Writing 10_mapper.py


**Proceso REDUCE 10**

In [32]:
%%writefile 10_reducer.py
#!/usr/bin/env python3
import sys
from collections import defaultdict

# utilizamos dos diccionarios
total_categoria = defaultdict(int)
nombres_categoria = defaultdict(list)

# por cada linea leemos la informacion del mapper
for linea in sys.stdin:
    contenido = linea.strip().split('\t')
    if len(contenido) != 3:
        continue
        
    categoria, numero, nombre = contenido
    numero = int(numero)
    
    # sumamos los totales
    total_categoria[categoria] += numero
    nombres_categoria[categoria].append(nombre)

#ordenamos por los valores de forma descendente (de mayor a menor)
total_ordenado = dict(sorted(total_categoria.items(), key=lambda item: item[1], reverse=True))

# imprimimos el numero de nombres por categoria
for categoria, nombre in total_ordenado.items():
    print(f"Hay {nombre} nombres {categoria}s")

# imprimimos los nombres largos
print("\nLos nombres largos son:\n")
for nombre in nombres_categoria['largo']:
    print(nombre)

Overwriting 10_reducer.py


In [33]:
# ejecutamos nuestro map-reduce y vemos el resultado desde la consola 
! cat /media/notebooks/argentina/convocatorias-2020.csv \
| python3 10_mapper.py | sort | python3 10_reducer.py > salida-10

! cat salida-10

Hay 9026 nombres medios
Hay 4511 nombres cortos
Hay 12 nombres largos

Los nombres largos son:

ADQUIRIR GAS OIL BUQUE RANCHO PARA ABASTECER A LAS UNIDADES NAVALES QUE REALIZARAN PATRULLAS MARÍTIMAS EN LA ZONA MAR DEL PLATA
ADQUIRIR GAS OÍL NAVAL RÉGIMEN EXENTO (RANCHO) PARA ABASTECER A LAS UNIDADES NAVALES INTERVINIENTES EN LAS OPERACIONES DE PATRULLA EN LA MILLA 200 2019/2020.
ADQUIRIR RACIONES ALIMENTARIAS PARA DESTINOS PERTENECIENTES AL COMANDO DE ADIESTRAMIENTO DE LA ARMADA DEPENDIENTE DE IBPB
ADQUISICION DE INDUMENTARIA PARA EL PERSONAL BRIGADISTA DE INCENDIO Y CONTRATADO DE LA RESERVA NATURAL ESTRICTA SAN ANTONIO.: CAMISAS  PANTALONES Y CALZADOS.
ADQUISICIÓN DE JUEGOS Y JUGUETES DIDÁCTICOS PARA EL JARDÍN MATERNAL N° 1461 MARÍA CALDERÓN DE LA BARCA DE LA BASE AERONAVAL ALMIRANTE ZAR
ADQUISICIÓN DE PASAJES PARA TRANSPORTE VIA TERRESTRE DEL PERSONAL POR COMISIONES DEL SERVICIO DE LOS DESTINOS DEPENDIENTES DE LA FUERZA AERONAVAL Nº 3 EN LA ZONA NAVAL 86
ADQUISICIÓN DE PINTURAS (Láte

**Ejecución Hadoop Streaming 10**

In [35]:
! hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-files 10_mapper.py,10_reducer.py \
-mapper 10_mapper.py \
-reducer 10_reducer.py \
-input /argentina/convocatorias-2020.csv \
-output /argentina/mapreduce/10_salida

packageJobJar: [/tmp/hadoop-unjar2049699401766062813/] [] /tmp/streamjob4499837957227080768.jar tmpDir=null
2024-08-28 08:44:28,299 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-28 08:44:28,872 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.7:8032
2024-08-28 08:44:29,726 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1724833657149_0009
2024-08-28 08:44:31,090 INFO mapred.FileInputFormat: Total input files to process : 1
2024-08-28 08:44:31,434 INFO mapreduce.JobSubmitter: number of splits:2
2024-08-28 08:44:31,873 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1724833657149_0009
2024-08-28 08:44:31,873 INFO mapreduce.JobSubmitter: Executing with tokens: []
2024-08-28 08:44:32,175 INFO conf.Configuration: resource-types.xml not found
2024-08-28 08:44:32,176 INFO resource.ResourceUtils

In [36]:
# Visualizamos la salida del ejercicio 10
! hdfs dfs -cat /argentina/mapreduce/10_salida/part-00000 

Hay 9026 nombres medios	
Hay 4511 nombres cortos	
Hay 12 nombres largos	
	
Los nombres largos son:	
	
Servicio de adicionales de la Policía para Puestos de Control de Pesos y Dimensiones y Campamentos del 21° Distrito	
ADQUISICIÓN DE JUEGOS Y JUGUETES DIDÁCTICOS PARA EL JARDÍN MATERNAL N° 1461 MARÍA CALDERÓN DE LA BARCA DE LA BASE AERONAVAL ALMIRANTE ZAR	
Provisión integral de raciones alimentarias para el personal militar de la Base naval PUERTO BELGRANO	
Adquisición de Pan Dulce con frutas abrillantadas y pasas de uva y Pan Dulce veneciano o madrileño (sin fruta).	
ADQUIRIR GAS OIL BUQUE RANCHO PARA ABASTECER A LAS UNIDADES NAVALES QUE REALIZARAN PATRULLAS MARÍTIMAS EN LA ZONA MAR DEL PLATA	
ADQUISICIÓN DE PINTURAS (Látex  esmaltes sintéticos  entonadores  diluyentes  enduidos  impregnantes  masillas  tintas colorantes  barniz  etc)	
ADQUIRIR GAS OÍL NAVAL RÉGIMEN EXENTO (RANCHO) PARA ABASTECER A LAS UNIDADES NAVALES INTERVINIENTES EN LAS OPERACIONES DE PATRULLA EN LA MILLA 200 2019/

**Comparación entre la salida obtenida y la esperada**

In [21]:
# funcion para la clasificación
def clasificar_longitud(longitud):
    if longitud <= 50:
        return 'corto'
    elif longitud <= 100:
        return 'medio'
    else:
        return 'largo'

In [22]:
# calculamos la longitud
df_convocatorias['longitud_nombre'] = df_convocatorias['nombre_del_procedimiento'].apply(len)

# clasificamos la nueva columna
df_convocatorias['categoria'] = df_convocatorias['longitud_nombre'].apply(clasificar_longitud)

# contamos el total para cada categoría
conteo_por_categoria = df_convocatorias['categoria'].value_counts()
print(conteo_por_categoria)

# nos quedamos con los nombres largos
nombres_largos = df_convocatorias[df_convocatorias['categoria'] == 'largo']['nombre_del_procedimiento']

# imprimimos los nombres largos
print("\nNombres de procedimientos largos:\n")
for nombre in nombres_largos:
    print(nombre)

categoria
medio    9026
corto    4511
largo      12
Name: count, dtype: int64

Nombres de procedimientos largos:

ADQUIRIR GAS OÍL NAVAL RÉGIMEN EXENTO (RANCHO) PARA ABASTECER A LAS UNIDADES NAVALES INTERVINIENTES EN LAS OPERACIONES DE PATRULLA EN LA MILLA 200 2019/2020.
ADQUIRIR GAS OIL BUQUE RANCHO PARA ABASTECER A LAS UNIDADES NAVALES QUE REALIZARAN PATRULLAS MARÍTIMAS EN LA ZONA MAR DEL PLATA
CONTRATACIÓN DE LICENCIAS ANTIVIRUS Y MONITOREO PARA LA DIRECCIÓN DE CIBERDEFENSA DEL EJÉRCITO ARGENTINO POR EL TÉRMINO DE UN AÑO CON SERVICIO DE MANTENIMIENTO
ADQUISICIÓN DE PASAJES PARA TRANSPORTE VIA TERRESTRE DEL PERSONAL POR COMISIONES DEL SERVICIO DE LOS DESTINOS DEPENDIENTES DE LA FUERZA AERONAVAL Nº 3 EN LA ZONA NAVAL 86
ADQUISICION DE INDUMENTARIA PARA EL PERSONAL BRIGADISTA DE INCENDIO Y CONTRATADO DE LA RESERVA NATURAL ESTRICTA SAN ANTONIO.: CAMISAS  PANTALONES Y CALZADOS.
ADQUIRIR RACIONES ALIMENTARIAS PARA DESTINOS PERTENECIENTES AL COMANDO DE ADIESTRAMIENTO DE LA ARMADA DEPENDIEN